<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c42-header.png" alt="Nextia Learning · Vision AI: See, Count and Inspect" width="100%">

# Solution: The image service and its review queue

**[The image service and its review queue](https://learning.nextia-ai.com/courses/vision/m08/the-image-service-and-its-review-queue/)** · Vision AI: See, Count and Inspect · Module 8, lesson 3 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** build the decision step of an image service by hand: a score, a zone, a confidence, a fallback for photos that cannot be judged, and a review folder for a person. Then you see which photos a person checks, and why labels from those photos alone are biased.

| | |
|---|---|
| **Time** | About 35 minutes |
| **Needs** | An internet connection for the setup cells (the returns photos and the service's inspector, 18 MB). No account and no key. |
| **You should know** | Pass, check and fail lines, from [Set the line with the warehouse](https://learning.nextia-ai.com/courses/vision/m06/set-the-line-with-the-warehouse/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, pandas 3.1.1, Pillow 13.0.0, scikit-learn 1.9.1 and matplotlib 3.11.2 and ONNX Runtime 1.31.1, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/vision/m08/the-image-service-and-its-review-queue/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C42/M08-L03-the-image-service-and-its-review-queue/the-image-service-and-its-review-queue-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, pandas 3.1.1, pillow 13.0.0, scikit-learn 1.9.1, scikit-image 0.26.0, onnxruntime 1.31.1, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 pandas==3.1.1 pillow==13.0.0 scikit-learn==1.9.1 scikit-image==0.26.0 matplotlib==3.11.2 onnxruntime==1.31.1
import warnings; warnings.filterwarnings("ignore")
import numpy; print("numpy", numpy.__version__)
import pandas; print("pandas", pandas.__version__)
import PIL; print("pillow", PIL.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)
import skimage; print("scikit-image", skimage.__version__)
import onnxruntime; print("onnxruntime", onnxruntime.__version__)

## Setup: get the Larkfield images

Run the next cell. It downloads the image sets that this lesson needs (about 34 MB, as zip files) into a folder `larkfield-images`, with the course's helper `compare.py`, checks each file's checksum and unpacks it. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-images` (in Colab: the **Files** panel on the left), then run the cell again.

Where the photos come from: the catalogue photos are cut from Open Images photos (CC BY 2.0: the author of each photo is in `data/catalogue/manifest.csv`), the returned candles are photos of the VisA inspection set (CC BY 4.0), and the pallets are drawn by our code. Read `data/dataset-card.md` for the details.

In [ ]:
# Setup: download the image sets into larkfield-images/, check them and unpack them.
import hashlib, os, shutil, subprocess, urllib.request, zipfile
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C42/"
FILES = {
    "archives/returns.zip": "dcf02f521f68228d34717b303628b2db8b19d18433126da2ce64047f1449210f",
    "compare.py": "e9794acacbce00c240906cc8b3e2256b8505ff4a8074b7507a8b159cd39b4826",
    "models/inspector/model.onnx": "82532b5a04f2245fc931e6548167eb72e36cfb9155a66532fc07a73e18bb6c7f",
    "models/inspector/bank.npy": "819fafe56cae7735557d7b0e0b4ac712f5ae2b2ec5bbf6ab4c71384aaa46f3fd",
    "models/inspector/config.json": "0283f64275f572ba4e5bbf37c64187cbe3ce7fc96a7a0162438648244e64159a",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-images").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-images and run again.")
    if name.endswith(".zip"):
        with zipfile.ZipFile(path) as z:
            z.extractall(PROJECT / "data")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
n = sum(1 for p in Path("data").rglob("*") if p.is_file())
print(f"Ready: {n} files in larkfield-images/data.")

## Setup: helpers

The next cell loads NumPy as `np`, pandas as `pd`, Pillow's `Image`, matplotlib, and the course helper as `cp` (the comparison table). It defines `show(paths)`, which draws photos in a row, and the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: images, the comparison table, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import compare as cp
warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
CLASSES = ["bowl", "candle", "chair", "cushion", "lamp", "mug", "plant_pot", "vase"]

def show(paths, titles=None, size=2.2):
    """Draw photos in a row (at most 8)."""
    paths = list(paths)[:8]
    fig, axes = plt.subplots(1, len(paths), figsize=(size * len(paths), size))
    for i, (ax, p) in enumerate(zip(np.atleast_1d(axes), paths)):
        ax.imshow(Image.open(p)); ax.axis("off")
        if titles:
            ax.set_title(str(titles[i]), fontsize=8)
    plt.show()

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The service's inspector

The setup cell also downloaded the service's inspector into `models/inspector/`: a ResNet-18 to its third layer in ONNX int8 (2.9 MB), a memory of 20,000 patches of good candles (15 MB), and `config.json` with the two lines. The lines were chosen on the validation photos for Kwame's 100 checks a day.

In [ ]:
import onnxruntime as ort
config = json.load(open("models/inspector/config.json"))
options = ort.SessionOptions()
options.intra_op_num_threads = 1  # one thread, as the service
session = ort.InferenceSession("models/inspector/model.onnx", options, providers=["CPUExecutionProvider"])
memory = np.load("models/inspector/bank.npy").astype(np.float32)   # 20,000 patches of good candles
memory_sq = (memory ** 2).sum(1)
MEAN = np.array([0.485, 0.456, 0.406], np.float32)
STD = np.array([0.229, 0.224, 0.225], np.float32)

def score(img):
    """The small PatchCore: 784 patches, each compared with the nearest good patch; the score is the worst one."""
    a = np.asarray(img.convert("RGB").resize((224, 224), Image.BILINEAR), np.float32) / 255
    f = session.run(None, {"image": ((a - MEAN) / STD).transpose(2, 0, 1)[None].astype(np.float32)})[0][0]
    q = f.reshape(f.shape[0], -1).T
    d2 = (q ** 2).sum(1)[:, None] + memory_sq[None] - 2 * q @ memory.T
    return float(np.sqrt(max(d2.min(1).max(), 0.0)))

CHECK, FAIL = config["check_line"], config["fail_line"]
print("model", config["version"], "check line", round(CHECK, 3), "fail line", round(FAIL, 3))

## 2. Score, zone and confidence

> **Predict.** Three test photos: a good candle, a candle with extra wax, and a candle with a bent wick. Which zone does each one get?

In [ ]:
def zone(s):
    return "fail" if s >= FAIL else "check" if s >= CHECK else "pass"

def confidence(s):
    """How far the score is from the nearest line, as a share of that line (0 between the lines, at most 1)."""
    if s >= FAIL:
        return round(min(1.0, (s - FAIL) / FAIL), 3)
    if s < CHECK:
        return round(min(1.0, (CHECK - s) / CHECK), 3)
    return 0.0

for item in ["R-6a26ad64", "R-33640732", "R-1cc8cc10"]:
    s = score(Image.open(f"data/returns/test/{item}.jpg"))
    print(item, zone(s), round(s, 3), "confidence", confidence(s))
show([f"data/returns/test/{i}.jpg" for i in ["R-6a26ad64", "R-33640732", "R-1cc8cc10"]], ["good", "extra wax", "bent wick"], size=3)

> **Check.** pass (5.832, confidence 0.334), check (9.924, confidence 0.0), fail (19.445, confidence 0.796). The confidence is not a probability: it says how far the score is from the nearest line.

## 3. The fallback

Some photos cannot be judged: a black photo, a thumbnail, a file that is not a photo. The service does not guess. It answers "check" and says why.

In [ ]:
def photo_problem(img):
    if min(img.size) < 100:
        return f"photo too small ({img.size[0]} x {img.size[1]} pixels)"
    brightness = float(np.asarray(img.convert("L")).mean())
    if brightness < 25:
        return f"photo too dark (mean brightness {brightness:.0f} of 255)"
    return None

print(photo_problem(Image.new("RGB", (320, 291), (8, 8, 8))))
print(photo_problem(Image.new("RGB", (64, 48), (200, 200, 200))))
print(photo_problem(Image.open("data/returns/test/R-6a26ad64.jpg")))

## 4. The review folder

A photo goes to a person when it is unsure (between the lines) or when the fallback answered. The service writes a copy and the reasons into `review/<day>/`. (The next lesson adds the face finder: every face is blurred before a copy is written.)

In [ ]:
import shutil

def judge(img, item_id, day="today", review="review"):
    reasons, s = [], None
    problem = photo_problem(img)
    if problem:
        z, conf = "check", 0.0
        reasons.append("fallback: " + problem)
    else:
        s = score(img)
        z, conf = zone(s), confidence(s)
        if z == "check":
            reasons.append("unsure: the score is between the two lines")
    result = {"item_id": item_id, "zone": z, "score": None if s is None else round(s, 4), "confidence": conf,
              "review": bool(reasons), "reasons": reasons, "model_version": config["version"]}
    if reasons:
        folder = Path(review) / day
        folder.mkdir(parents=True, exist_ok=True)
        img.convert("RGB").save(folder / f"{item_id}.jpg", quality=90)
        (folder / f"{item_id}.json").write_text(json.dumps(result, indent=1))
    return result

shutil.rmtree("review", ignore_errors=True)
for item in ["R-6a26ad64", "R-33640732", "R-1cc8cc10"]:
    print(judge(Image.open(f"data/returns/test/{item}.jpg"), item))
print(judge(Image.new("RGB", (320, 291), (8, 8, 8)), "R-dark"))
print(sorted(str(p) for p in Path("review").rglob("*")))

## 5. Every validation and test photo

Now score the 330 validation and test photos (about 10 seconds) and measure the service on the test photos with the course's costs.

In [ ]:
returns = pd.read_csv("data/returns/manifest.csv").fillna("")
rows = returns[returns["split"].isin(["val", "test"])].reset_index(drop=True)
rows["score"] = [score(Image.open("data/" + f)) for f in rows["file"]]
rows["damaged"] = rows["status"] == "damaged"
rows["zone"] = [zone(s) for s in rows["score"]]
test = rows[rows["split"] == "test"]
print("AUROC on test:", cp.roc_auc(list(test["score"]), list(test["damaged"])))
print(cp.inspection_costs(list(test["score"]), list(test["damaged"]), CHECK, FAIL))

> **Check.** AUROC 0.836 and 988.1 dollars a day: the numbers of the service's inspector (int8). The same small PatchCore in 32-bit numbers had 0.847 and 1,290.8 in Module 6: int8 changed the scores a little, and the lines with them.

## 6. Who sees which photos

A person sees the check zone. Count the damaged photos in each zone of the test photos.

In [ ]:
test.groupby("zone")["damaged"].agg(photos="count", damaged="sum")

> **Check.** check: 32 photos, 3 damaged; fail: 9 photos, 8 damaged; pass: 177 photos.

## 7. Your turn: what the review folder never shows

> **Your turn.** Put the number of **damaged** test photos in the **pass** zone in `missed_in_pass` (an integer). These go back on sale, and no person sees them.

In [ ]:
missed_in_pass = int(((test["zone"] == "pass") & test["damaged"]).sum())
print(missed_in_pass)
print(test[(test["zone"] == "pass") & test["damaged"]][["item_id", "score", "damage"]])

In [ ]:
expect('missed_in_pass', missed_in_pass, 7)

If Priya builds the next set of labels only from the review folder, every new label has a score between the two lines. The damaged candles that pass (low scores: small colour spots, packet corners) never become labels, so the next lines cannot learn about them. The fix: also send a small **random sample** of passed and failed items to a person, and record why each photo was picked.

## Next

The [lesson page](https://learning.nextia-ai.com/courses/vision/m08/the-image-service-and-its-review-queue/) has the diagrams, the service itself (`vision serve`) and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [People in images](https://learning.nextia-ai.com/courses/vision/m08/people-in-images/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/vision/m08/the-image-service-and-its-review-queue/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The photos are open photos (each with its author and licence in the manifests) or images made by our code; no photo shows a real person.